# DL-01 — Prepare Raster Labels and Patches

This notebook prepares the deep learning dataset for the West Java biomass-derived carbon nanomaterial feedstock suitability study.

## Required files in Google Drive
Put these files in:

```text
/content/drive/MyDrive/geospatial_biomass_carbon/
```

```text
grid_suitability_jawa_barat_ml_prediction.gpkg
sentinel2_indices_jawa_barat_2024_scale50.tif
viirs_nighttime_lights_jawa_barat_2024.tif
cropland_mask_jawa_barat.tif
esa_worldcover_jawa_barat.tif
```

## Main workflow

```text
Load ML grid prediction GPKG
→ rasterize class and score labels
→ align raster inputs to Sentinel-2 reference grid
→ build multiband input stack
→ normalize input bands
→ extract 128 × 128 patches
→ save patch dataset for U-Net / DeepLabV3+
```

## Main outputs

```text
input_stack_geospatial_features_raw.tif
label_potential_class_final.tif
label_suitability_score_final.tif
patch_dataset_classification_regression.npz
patch_metadata.csv
band_normalization_stats.csv
dl01_summary.xlsx
```


## 1. Install required libraries

In [ ]:
!pip install geopandas rasterio pyogrio fiona openpyxl tqdm -q

## 2. Import libraries

In [ ]:
import os
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
import fiona
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)


## 3. Mount Google Drive and set paths

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/geospatial_biomass_carbon')
OUTPUT_DIR = BASE_DIR / 'deep_learning' / 'DL_01_prepare_raster_labels_and_patches'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_GPKG = BASE_DIR / 'grid_suitability_jawa_barat_ml_prediction.gpkg'
SENTINEL_FILE = BASE_DIR / 'sentinel2_indices_jawa_barat_2024_scale50.tif'
VIIRS_FILE = BASE_DIR / 'viirs_nighttime_lights_jawa_barat_2024.tif'
CROPLAND_FILE = BASE_DIR / 'cropland_mask_jawa_barat.tif'
LANDCOVER_FILE = BASE_DIR / 'esa_worldcover_jawa_barat.tif'

OUTPUT_INPUT_STACK = OUTPUT_DIR / 'input_stack_geospatial_features_raw.tif'
OUTPUT_CLASS_LABEL = OUTPUT_DIR / 'label_potential_class_final.tif'
OUTPUT_SCORE_LABEL = OUTPUT_DIR / 'label_suitability_score_final.tif'
OUTPUT_PATCH_NPZ = OUTPUT_DIR / 'patch_dataset_classification_regression.npz'
OUTPUT_PATCH_METADATA = OUTPUT_DIR / 'patch_metadata.csv'
OUTPUT_BAND_STATS = OUTPUT_DIR / 'band_normalization_stats.csv'
OUTPUT_SUMMARY_XLSX = OUTPUT_DIR / 'dl01_summary.xlsx'

required_files = [GRID_GPKG, SENTINEL_FILE, VIIRS_FILE, CROPLAND_FILE, LANDCOVER_FILE]
for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print('All required input files found.')
print('Output directory:', OUTPUT_DIR)


## 4. Configuration

In [ ]:
# Patch extraction settings
PATCH_SIZE = 128
STRIDE = 128
MIN_VALID_RATIO = 0.60
MAX_PATCHES = 2500
BALANCE_CLASSES = True

# Label source. Default uses rule-based final class as the deep learning label.
PREFERRED_CLASS_COLS = [
    'potential_class_final',
    'ml_predicted_potential_class',
    'potential_class_grid_without_road'
]

PREFERRED_SCORE_COLS = [
    'suitability_score_final',
    'ml_predicted_suitability_score',
    'xgb_predicted_score',
    'suitability_score_without_road'
]

# Tabular features from grid to rasterize as additional input channels if available.
TABULAR_RASTER_FEATURES = [
    'distance_to_road_km',
    'distance_to_agroindustry_km',
    'estimated_grid_residue_ton_year',
    'agroindustry_count_5km',
    'agroindustry_count_10km',
    'cropland_ratio',
    'environmental_score'
]

# Sentinel-2 multiband order from previous GEE export.
SENTINEL_BAND_NAMES = [
    'B2', 'B3', 'B4', 'B8', 'B11', 'B12',
    'NDVI', 'EVI', 'NDWI', 'NBR', 'SAVI'
]

CLASS_MAPPING = {
    'Very Low': 0,
    'Low': 1,
    'Moderate': 2,
    'High': 3,
    'Very High': 4
}
CLASS_NAMES = {v: k for k, v in CLASS_MAPPING.items()}
CLASS_NODATA = 255
SCORE_NODATA = -9999.0

print('PATCH_SIZE:', PATCH_SIZE)
print('STRIDE:', STRIDE)
print('MAX_PATCHES:', MAX_PATCHES)


## 5. Load grid prediction layer

In [ ]:
layers = fiona.listlayers(str(GRID_GPKG))
print('Available layers:', layers)
GRID_LAYER = 'grid_ml_prediction' if 'grid_ml_prediction' in layers else layers[0]
print('Using layer:', GRID_LAYER)

grid = gpd.read_file(GRID_GPKG, layer=GRID_LAYER)
print('Grid shape:', grid.shape)
print('Grid CRS:', grid.crs)
print('Columns:')
print(grid.columns.tolist())

if grid.empty:
    raise ValueError('Grid layer is empty.')

grid.head()


## 6. Select label columns and prepare class IDs

In [ ]:
def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

CLASS_COL = first_existing_column(grid, PREFERRED_CLASS_COLS)
SCORE_COL = first_existing_column(grid, PREFERRED_SCORE_COLS)

if CLASS_COL is None:
    raise ValueError(f'No class label column found. Candidates: {PREFERRED_CLASS_COLS}')
if SCORE_COL is None:
    raise ValueError(f'No suitability score column found. Candidates: {PREFERRED_SCORE_COLS}')

print('Selected class column:', CLASS_COL)
print('Selected score column:', SCORE_COL)

grid['dl_class_name'] = grid[CLASS_COL].astype(str).str.strip()
grid['dl_class_id'] = grid['dl_class_name'].map(CLASS_MAPPING)

missing_class = grid[grid['dl_class_id'].isna()][['dl_class_name']].drop_duplicates()
if len(missing_class) > 0:
    display(missing_class)
    raise ValueError('Unmapped class labels were found. Update CLASS_MAPPING.')

grid['dl_class_id'] = grid['dl_class_id'].astype(np.uint8)
grid['dl_score'] = pd.to_numeric(grid[SCORE_COL], errors='coerce').astype('float32')

print('Class distribution:')
display(grid['dl_class_name'].value_counts().reindex(CLASS_MAPPING.keys()))

print('Score summary:')
display(grid['dl_score'].describe())


## 7. Use Sentinel-2 as reference raster

In [ ]:
with rasterio.open(SENTINEL_FILE) as src:
    ref_profile = src.profile.copy()
    ref_crs = src.crs
    ref_transform = src.transform
    ref_width = src.width
    ref_height = src.height
    sentinel_count = src.count
    sentinel_nodata = src.nodata

print('Reference CRS:', ref_crs)
print('Width x Height:', ref_width, 'x', ref_height)
print('Sentinel band count:', sentinel_count)
print('Sentinel nodata:', sentinel_nodata)

if sentinel_count != len(SENTINEL_BAND_NAMES):
    raise ValueError(f'Sentinel band count ({sentinel_count}) does not match expected bands ({len(SENTINEL_BAND_NAMES)}).')


## 8. Rasterize grid labels and tabular features

In [ ]:
grid_ref = grid.to_crs(ref_crs)

# Rasterize class label
class_shapes = [(geom, int(value)) for geom, value in zip(grid_ref.geometry, grid_ref['dl_class_id']) if geom is not None]
class_label = rasterize(
    class_shapes,
    out_shape=(ref_height, ref_width),
    transform=ref_transform,
    fill=CLASS_NODATA,
    dtype='uint8',
    all_touched=True
)

# Rasterize suitability score label
score_shapes = [
    (geom, float(value))
    for geom, value in zip(grid_ref.geometry, grid_ref['dl_score'])
    if geom is not None and np.isfinite(value)
]
score_label = rasterize(
    score_shapes,
    out_shape=(ref_height, ref_width),
    transform=ref_transform,
    fill=SCORE_NODATA,
    dtype='float32',
    all_touched=True
)

# Save label rasters
class_profile = ref_profile.copy()
class_profile.update(count=1, dtype='uint8', nodata=CLASS_NODATA, compress='deflate')
with rasterio.open(OUTPUT_CLASS_LABEL, 'w', **class_profile) as dst:
    dst.write(class_label, 1)

score_profile = ref_profile.copy()
score_profile.update(count=1, dtype='float32', nodata=SCORE_NODATA, compress='deflate')
with rasterio.open(OUTPUT_SCORE_LABEL, 'w', **score_profile) as dst:
    dst.write(score_label, 1)

print('Saved:', OUTPUT_CLASS_LABEL)
print('Saved:', OUTPUT_SCORE_LABEL)

# Rasterize tabular features from grid polygons.
rasterized_tabular = {}
for col in TABULAR_RASTER_FEATURES:
    if col not in grid_ref.columns:
        print(f'Skipping missing tabular feature: {col}')
        continue
    vals = pd.to_numeric(grid_ref[col], errors='coerce')
    shapes = [(geom, float(v)) for geom, v in zip(grid_ref.geometry, vals) if geom is not None and np.isfinite(v)]
    arr = rasterize(
        shapes,
        out_shape=(ref_height, ref_width),
        transform=ref_transform,
        fill=np.nan,
        dtype='float32',
        all_touched=True
    )
    rasterized_tabular[col] = arr

print('Rasterized tabular features:', list(rasterized_tabular.keys()))


## 9. Align external rasters to Sentinel-2 grid

In [ ]:
def read_and_align_single_band(path, resampling=Resampling.bilinear, dst_dtype='float32'):
    # Read a single-band raster and align it to the Sentinel-2 reference grid.
    path = Path(path)
    with rasterio.open(path) as src:
        src_arr = src.read(1)
        src_nodata = src.nodata
        dst = np.empty((ref_height, ref_width), dtype=dst_dtype)
        if np.issubdtype(np.dtype(dst_dtype), np.floating):
            dst[:] = np.nan
        else:
            dst[:] = 0

        same_grid = (
            src.crs == ref_crs and
            src.transform == ref_transform and
            src.width == ref_width and
            src.height == ref_height
        )

        if same_grid:
            arr = src_arr.astype(dst_dtype)
        else:
            reproject(
                source=src_arr,
                destination=dst,
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=src_nodata,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                dst_nodata=np.nan if np.issubdtype(np.dtype(dst_dtype), np.floating) else 0,
                resampling=resampling
            )
            arr = dst

        arr = arr.astype(dst_dtype)
        if src_nodata is not None and np.issubdtype(arr.dtype, np.floating):
            arr[arr == src_nodata] = np.nan
        return arr

# Read Sentinel-2 stack
with rasterio.open(SENTINEL_FILE) as src:
    sentinel_stack = src.read().astype('float32')
    if src.nodata is not None:
        sentinel_stack[sentinel_stack == src.nodata] = np.nan

viirs_arr = read_and_align_single_band(VIIRS_FILE, resampling=Resampling.bilinear, dst_dtype='float32')
cropland_arr = read_and_align_single_band(CROPLAND_FILE, resampling=Resampling.nearest, dst_dtype='float32')
landcover_arr = read_and_align_single_band(LANDCOVER_FILE, resampling=Resampling.nearest, dst_dtype='float32')

cropland_arr = np.clip(cropland_arr, 0, 1)

print('Sentinel stack:', sentinel_stack.shape)
print('VIIRS:', viirs_arr.shape)
print('Cropland:', cropland_arr.shape)
print('Landcover:', landcover_arr.shape)


## 10. Build and save multiband input stack

In [ ]:
input_arrays = []
band_names = []

for i, name in enumerate(SENTINEL_BAND_NAMES):
    input_arrays.append(sentinel_stack[i])
    band_names.append(name)

input_arrays.append(viirs_arr)
band_names.append('VIIRS_NTL')

input_arrays.append(cropland_arr)
band_names.append('cropland_mask')

input_arrays.append(landcover_arr)
band_names.append('ESA_WorldCover')

for col, arr in rasterized_tabular.items():
    input_arrays.append(arr.astype('float32'))
    band_names.append(col)

input_stack_raw = np.stack(input_arrays, axis=0).astype('float32')

print('Input stack shape:', input_stack_raw.shape)
for i, b in enumerate(band_names, start=1):
    print(i, b)

stack_profile = ref_profile.copy()
stack_profile.update(count=len(band_names), dtype='float32', nodata=np.nan, compress='deflate')

with rasterio.open(OUTPUT_INPUT_STACK, 'w', **stack_profile) as dst:
    dst.write(input_stack_raw)
    for i, b in enumerate(band_names, start=1):
        dst.set_band_description(i, b)

print('Saved raw input stack:', OUTPUT_INPUT_STACK)


## 11. Normalize input stack

In [ ]:
valid_label_mask = class_label != CLASS_NODATA
input_stack_norm = np.empty_like(input_stack_raw, dtype='float32')
stats_records = []

for i, name in enumerate(band_names):
    arr = input_stack_raw[i].astype('float32')
    valid = np.isfinite(arr) & valid_label_mask

    if name == 'ESA_WorldCover':
        norm = arr / 100.0
        norm[~np.isfinite(norm)] = 0
        method = 'scale_100'
        mean_val = float(np.nanmean(arr[valid])) if valid.any() else np.nan
        std_val = float(np.nanstd(arr[valid])) if valid.any() else np.nan
    else:
        mean_val = float(np.nanmean(arr[valid])) if valid.any() else 0.0
        std_val = float(np.nanstd(arr[valid])) if valid.any() else 1.0
        if std_val == 0 or not np.isfinite(std_val):
            std_val = 1.0
        norm = (arr - mean_val) / std_val
        norm[~np.isfinite(norm)] = 0
        method = 'zscore'

    input_stack_norm[i] = norm.astype('float32')
    stats_records.append({
        'band_index': i + 1,
        'band_name': name,
        'normalization': method,
        'mean': mean_val,
        'std': std_val,
        'min_raw': float(np.nanmin(arr[valid])) if valid.any() else np.nan,
        'max_raw': float(np.nanmax(arr[valid])) if valid.any() else np.nan
    })

band_stats = pd.DataFrame(stats_records)
band_stats.to_csv(OUTPUT_BAND_STATS, index=False)
display(band_stats)
print('Saved:', OUTPUT_BAND_STATS)


## 12. Visual quality check

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
ndvi_idx = band_names.index('NDVI')

axes[0,0].imshow(input_stack_raw[ndvi_idx], cmap='YlGn')
axes[0,0].set_title('NDVI')
axes[0,1].imshow(viirs_arr, cmap='magma')
axes[0,1].set_title('VIIRS Nighttime Lights')
axes[0,2].imshow(cropland_arr, cmap='Greens', vmin=0, vmax=1)
axes[0,2].set_title('Cropland Mask')
axes[1,0].imshow(landcover_arr, cmap='tab20')
axes[1,0].set_title('ESA WorldCover')
axes[1,1].imshow(np.where(class_label == CLASS_NODATA, np.nan, class_label), cmap='RdYlGn_r', vmin=0, vmax=4)
axes[1,1].set_title('Class Label')
axes[1,2].imshow(np.where(score_label == SCORE_NODATA, np.nan, score_label), cmap='inferno')
axes[1,2].set_title('Suitability Score')

for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()


## 13. Scan candidate patches

In [ ]:
H, W = class_label.shape
P = PATCH_SIZE
S = STRIDE
patch_records = []

for row in tqdm(range(0, H - P + 1, S), desc='Scanning rows'):
    for col in range(0, W - P + 1, S):
        y_patch = class_label[row:row+P, col:col+P]
        valid = y_patch != CLASS_NODATA
        valid_ratio = valid.mean()
        if valid_ratio < MIN_VALID_RATIO:
            continue
        classes = y_patch[valid]
        if len(classes) == 0:
            continue
        counts = Counter(classes.tolist())
        dominant_class = max(counts, key=counts.get)
        dominant_fraction = counts[dominant_class] / len(classes)
        patch_records.append({
            'row': row,
            'col': col,
            'valid_ratio': valid_ratio,
            'dominant_class_id': int(dominant_class),
            'dominant_class_name': CLASS_NAMES[int(dominant_class)],
            'dominant_fraction': dominant_fraction
        })

patch_df = pd.DataFrame(patch_records)
print('Candidate patches:', len(patch_df))
if len(patch_df) == 0:
    raise ValueError('No candidate patches found. Reduce MIN_VALID_RATIO or PATCH_SIZE.')

display(patch_df['dominant_class_name'].value_counts())
display(patch_df.head())


## 14. Select patches

In [ ]:
if MAX_PATCHES is not None and len(patch_df) > MAX_PATCHES:
    if BALANCE_CLASSES:
        selected_parts = []
        classes = sorted(patch_df['dominant_class_id'].unique())
        per_class = max(1, MAX_PATCHES // len(classes))
        for cls in classes:
            part = patch_df[patch_df['dominant_class_id'] == cls]
            n = min(per_class, len(part))
            selected_parts.append(part.sample(n=n, random_state=42))
        selected_df = pd.concat(selected_parts, ignore_index=True)
        if len(selected_df) < MAX_PATCHES:
            rest = patch_df.drop(selected_df.index, errors='ignore')
            remaining = MAX_PATCHES - len(selected_df)
            if len(rest) > 0:
                selected_df = pd.concat([selected_df, rest.sample(n=min(remaining, len(rest)), random_state=42)], ignore_index=True)
    else:
        selected_df = patch_df.sample(n=MAX_PATCHES, random_state=42).reset_index(drop=True)
else:
    selected_df = patch_df.reset_index(drop=True)

selected_df = selected_df.reset_index(drop=True)
selected_df['patch_id'] = [f'patch_{i:05d}' for i in range(len(selected_df))]

print('Selected patches:', len(selected_df))
display(selected_df['dominant_class_name'].value_counts())


## 15. Extract and save patch dataset

In [ ]:
N = len(selected_df)
B = input_stack_norm.shape[0]

X_patches = np.empty((N, B, P, P), dtype=np.float16)
y_class_patches = np.empty((N, P, P), dtype=np.uint8)
y_score_patches = np.empty((N, P, P), dtype=np.float32)

for idx, rec in tqdm(selected_df.iterrows(), total=N, desc='Extracting patches'):
    row = int(rec['row'])
    col = int(rec['col'])
    X_patches[idx] = input_stack_norm[:, row:row+P, col:col+P].astype(np.float16)
    y_class_patches[idx] = class_label[row:row+P, col:col+P]
    y_score_patches[idx] = score_label[row:row+P, col:col+P]

selected_df['patch_size'] = PATCH_SIZE
selected_df['stride'] = STRIDE
selected_df['class_nodata'] = CLASS_NODATA
selected_df['score_nodata'] = SCORE_NODATA
selected_df['source_class_column'] = CLASS_COL
selected_df['source_score_column'] = SCORE_COL
selected_df.to_csv(OUTPUT_PATCH_METADATA, index=False)

np.savez_compressed(
    OUTPUT_PATCH_NPZ,
    X=X_patches,
    y_class=y_class_patches,
    y_score=y_score_patches,
    band_names=np.array(band_names),
    class_mapping=json.dumps(CLASS_MAPPING),
    class_nodata=np.array(CLASS_NODATA),
    score_nodata=np.array(SCORE_NODATA)
)

print('Saved patch dataset:', OUTPUT_PATCH_NPZ)
print('Saved metadata:', OUTPUT_PATCH_METADATA)
print('X shape:', X_patches.shape)
print('y_class shape:', y_class_patches.shape)
print('y_score shape:', y_score_patches.shape)
print('NPZ size MB:', OUTPUT_PATCH_NPZ.stat().st_size / (1024**2))


## 16. Patch preview

In [ ]:
n_preview = min(3, len(selected_df))
preview_indices = np.random.choice(len(selected_df), n_preview, replace=False)

for pi in preview_indices:
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    ndvi = X_patches[pi, band_names.index('NDVI')].astype('float32')
    viirs = X_patches[pi, band_names.index('VIIRS_NTL')].astype('float32')
    yc = y_class_patches[pi]
    ys = y_score_patches[pi]

    axes[0].imshow(ndvi, cmap='YlGn')
    axes[0].set_title('Normalized NDVI')
    axes[1].imshow(viirs, cmap='magma')
    axes[1].set_title('Normalized VIIRS')
    axes[2].imshow(np.where(yc == CLASS_NODATA, np.nan, yc), cmap='RdYlGn_r', vmin=0, vmax=4)
    axes[2].set_title('Class label')
    axes[3].imshow(np.where(ys == SCORE_NODATA, np.nan, ys), cmap='inferno')
    axes[3].set_title('Score label')

    for ax in axes:
        ax.axis('off')
    plt.suptitle(f"{selected_df.loc[pi, 'patch_id']} | dominant: {selected_df.loc[pi, 'dominant_class_name']}")
    plt.tight_layout()
    plt.show()


## 17. Save DL-01 summary

In [ ]:
summary_main = pd.DataFrame([
    {'item': 'grid_file', 'value': str(GRID_GPKG)},
    {'item': 'grid_layer', 'value': GRID_LAYER},
    {'item': 'sentinel_reference_file', 'value': str(SENTINEL_FILE)},
    {'item': 'reference_crs', 'value': str(ref_crs)},
    {'item': 'reference_width', 'value': ref_width},
    {'item': 'reference_height', 'value': ref_height},
    {'item': 'input_band_count', 'value': len(band_names)},
    {'item': 'class_column', 'value': CLASS_COL},
    {'item': 'score_column', 'value': SCORE_COL},
    {'item': 'candidate_patches', 'value': len(patch_df)},
    {'item': 'selected_patches', 'value': len(selected_df)},
    {'item': 'patch_size', 'value': PATCH_SIZE},
    {'item': 'stride', 'value': STRIDE},
    {'item': 'min_valid_ratio', 'value': MIN_VALID_RATIO},
    {'item': 'max_patches', 'value': MAX_PATCHES},
    {'item': 'patch_dataset', 'value': str(OUTPUT_PATCH_NPZ)},
])

band_table = pd.DataFrame({'band_index': range(1, len(band_names)+1), 'band_name': band_names})
class_table = pd.DataFrame([{'class_id': v, 'class_name': k} for k, v in CLASS_MAPPING.items()])
patch_class_dist = selected_df['dominant_class_name'].value_counts().reset_index()
patch_class_dist.columns = ['dominant_class_name', 'patch_count']
output_files = pd.DataFrame([
    {'output_file': 'input_stack_geospatial_features_raw.tif', 'path': str(OUTPUT_INPUT_STACK)},
    {'output_file': 'label_potential_class_final.tif', 'path': str(OUTPUT_CLASS_LABEL)},
    {'output_file': 'label_suitability_score_final.tif', 'path': str(OUTPUT_SCORE_LABEL)},
    {'output_file': 'patch_dataset_classification_regression.npz', 'path': str(OUTPUT_PATCH_NPZ)},
    {'output_file': 'patch_metadata.csv', 'path': str(OUTPUT_PATCH_METADATA)},
    {'output_file': 'band_normalization_stats.csv', 'path': str(OUTPUT_BAND_STATS)},
])

with pd.ExcelWriter(OUTPUT_SUMMARY_XLSX, engine='openpyxl') as writer:
    summary_main.to_excel(writer, sheet_name='DL01 Summary', index=False)
    band_table.to_excel(writer, sheet_name='Input Bands', index=False)
    class_table.to_excel(writer, sheet_name='Class Mapping', index=False)
    patch_class_dist.to_excel(writer, sheet_name='Patch Class Dist', index=False)
    band_stats.to_excel(writer, sheet_name='Normalization Stats', index=False)
    output_files.to_excel(writer, sheet_name='Output Files', index=False)

print('Saved summary:', OUTPUT_SUMMARY_XLSX)
display(summary_main)
display(output_files)


## 18. Optional download summary files

In [ ]:
from google.colab import files

print('Important outputs are saved in:')
print(OUTPUT_DIR)

# The patch dataset can be large, so keep it in Google Drive.
# Uncomment these if you want direct downloads of the small summary files.
# files.download(str(OUTPUT_SUMMARY_XLSX))
# files.download(str(OUTPUT_PATCH_METADATA))
# files.download(str(OUTPUT_BAND_STATS))
